In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pickle

from scripts.simulation.sequence import simulate_sequences, downsample_sequences
import scripts.simulation.rank_correlation as rs 
from scripts.simulation.correlation_mean import add_within_clust_score
nrm = np.load("scripts/simulation/nrm.npy")

In [2]:
# PARAMETERS
params = {
    "n_neurons": 100,
    "n_motifs": 20,
    "n_bins": 100,
    "n_sequences": 100,
    "sigma_range": (1,1),    # increasing range makes sequences less similar
    "vol_param": (0.07, 0.9),     # beta distribution:  0.07, 0.9
                                  # a inactive (lower -> more neurons inactive, if high longer seqs, mean len shifts)
                                  # b active (lower -> more neurons active, if <1, skewed to 1, if lower longer seqs, mean len shifts)
    "corr_mu": False,
    "rho_mu": 0.0,
    "corr_sigma": False,
    "rho_sig": 0.0,
    "corr_volume": False,
    "rho_vol": 0.0,
    "shuffle_order": False,       # shuffle order of sequences
}

In [ ]:
seqs, seqs_labels, spk_times, sequences, true_templates, mu, sigma, volume, densities, cdfs = simulate_sequences(**params, random_state=1, plot=False, batch_size=5000)

In [ ]:
seqs_downsampled, spk_times_downsampled, volume_downsampled = downsample_sequences(sequences, spk_times, volume, params["n_neurons"], n_neurons_keep=100, min_length=5, random_state=1)

In [ ]:
rep_index, nsig, pval, bmat, zmat, corrmat = rs.allmot(seqs, nrm)

In [ ]:
within_score, within_var = add_within_clust_score(seqs_labels, zmat, bmat)

# Create proper mapping from unique motif IDs to scores
unique_motifs = np.unique(seqs_labels)
motif_to_score = {motif: score for motif, score in zip(unique_motifs, within_score)}
motif_to_var = {motif: var for motif, var in zip(unique_motifs, within_var)}

# Sort by highest score first (excluding NaN)
valid_motifs = [m for m in unique_motifs if not np.isnan(motif_to_score[m])]
sorted_motifs = sorted(valid_motifs, key=lambda m: motif_to_score[m], reverse=True)
scores_sorted = [motif_to_score[m] for m in sorted_motifs]
vars_sorted = [motif_to_var[m] for m in sorted_motifs]

In [ ]:
seed_scores = {}
for seed in range(50):
    seqs, seqs_labels, _, _, _, _, _, _, _, _ = simulate_sequences(**params, random_state=seed, plot=False, batch_size=5000)
    rep_index, nsig, pval, bmat, zmat, corrmat = rs.allmot(seqs, nrm)
    within_score, within_var = add_within_clust_score(seqs_labels, zmat, bmat)
    unique_motifs = np.unique(seqs_labels)
    motif_to_score = {motif: score for motif, score in zip(unique_motifs, within_score)}
    seed_scores[seed] = motif_to_score
    del seqs, seqs_labels, rep_index, nsig, pval, bmat, zmat, corrmat

with open(f'50seed_orig_zscore_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump(seed_scores, f)

In [10]:
## SAVING THE ORIGINAL N=100 SEQUENCES, SPIKE TIMES, VOLUMES, AND LABELS FOR EACH SEED ##
seed_values= {}

for seed in range(50):
    seqs, seqs_labels, spk_times, _, _, _, _, _, _, _ = simulate_sequences(**params, random_state=seed, plot=False, batch_size=5000)
    seed_values[seed] = {
        'seqs': seqs,
        'seqs_labels': seqs_labels,
        'spike_times': spk_times
    }
    print(f"Seed {seed} processed.")
    del seqs, seqs_labels, spk_times

with open(f'50seed_orig_seq_spk_label_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump(seed_values, f)

Seed 0 processed.
Seed 1 processed.
Seed 2 processed.
Seed 3 processed.
Seed 4 processed.
Seed 5 processed.
Seed 6 processed.
Seed 7 processed.
Seed 8 processed.
Seed 9 processed.
Seed 10 processed.
Seed 11 processed.
Seed 12 processed.
Seed 13 processed.
Seed 14 processed.
Seed 15 processed.
Seed 16 processed.
Seed 17 processed.
Seed 18 processed.
Seed 19 processed.
Seed 20 processed.
Seed 21 processed.
Seed 22 processed.
Seed 23 processed.
Seed 24 processed.
Seed 25 processed.
Seed 26 processed.
Seed 27 processed.
Seed 28 processed.
Seed 29 processed.
Seed 30 processed.
Seed 31 processed.
Seed 32 processed.
Seed 33 processed.
Seed 34 processed.
Seed 35 processed.
Seed 36 processed.
Seed 37 processed.
Seed 38 processed.
Seed 39 processed.
Seed 40 processed.
Seed 41 processed.
Seed 42 processed.
Seed 43 processed.
Seed 44 processed.
Seed 45 processed.
Seed 46 processed.
Seed 47 processed.
Seed 48 processed.
Seed 49 processed.


In [ ]:
with open(f'50seed_orig_zscore_100_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    seed_scores_100 = pickle.load(f)

In [ ]:
all_scores_orig = []
for seed, motif_dict in seed_scores_100.items():
    for motif_id, score in motif_dict.items():
        if not np.isnan(score):  # exclude NaN values
            all_scores_orig.append(score)

all_scores_orig = np.array(all_scores_orig)

# sum the number of scores above 0.4
threshold = 0.4
num_above_threshold_orig = np.sum(all_scores_orig > threshold)
print(f"Number of scores above {threshold}: {num_above_threshold_orig}")

plt.figure(figsize=(8, 5))
plt.hist(all_scores_orig, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Within-Cluster Score')
plt.ylabel('Count')
plt.axvline(x=0.4, color='red', linestyle='--', label=f'{num_above_threshold_orig} motif score above {threshold}')
plt.title(f'Distribution of All Motif Scores (50 seeds, {len(all_scores_orig)} motifs total)\nsigma={params["sigma_range"]}, vol={params["vol_param"]}')
plt.legend()
#plt.savefig(f'50seed_orig_zscore_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}_histogram.jpg', dpi=300)
#plt.savefig(f'50seed_orig_zscore_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}_histogram.pdf', dpi=300)
plt.show()

print(f"Total scores: {len(all_scores_orig)}")
print(f"Mean: {all_scores_orig.mean():.4f}")
print(f"Std: {all_scores_orig.std():.4f}")
print(f"Min: {all_scores_orig.min():.4f}")
print(f"Max: {all_scores_orig.max():.4f}")


In [ ]:
## SAVING THE DOWNSAMPLED SEQUENCES, SPIKE TIMES, VOLUMES, AND LABELS FOR EACH SEED ##
seed_scores = {}
params["n_neurons"] = 100000

volumes = [(0.07, 0.9), (0.07, 0.5), (0.1, 0.5)]

for vol_param in volumes:
    params["vol_param"] = vol_param
    for seed in range(50):
        seqs, seqs_labels, spk_times, sequences, true_templates, mu, sigma, volume, _, _ = simulate_sequences(**params, random_state=seed, plot=False, batch_size=5000)
        seqs_downsampled, spk_times_downsampled, volume_downsampled = downsample_sequences(sequences, spk_times, volume, params["n_neurons"], n_neurons_keep=100, min_length=5, random_state=1)

        # rebuild labels for downsampled sequences using original motif labels from simulation
        original_label_set = set(seqs_labels)
        seqs_labels_downsampled= [
            motif_id
            for motif_id, seq_list in seqs_downsampled.items()
            for _ in seq_list
        ]

        motif_idx_down = []
        motif_ids = [m for m in seqs_downsampled.keys()] # if len(seqs_downsampled[m]) > 20] 

        seqs_flat = []
        for motif_id in motif_ids:
            motif_idx_down.extend([motif_id] * len(seqs_downsampled[motif_id]))
            seqs_flat.extend(seqs_downsampled[motif_id])
        
        seed_scores[seed] = {
            'seqs': seqs_downsampled,
            'seqs_labels': seqs_labels_downsampled,
            'spike_times': spk_times_downsampled,
            'volume': volume_downsampled
        }
        
        print(f"Seed {seed} done.")

        # empty the variables to save memory
        del seqs, seqs_labels, spk_times, seqs_downsampled, spk_times_downsampled, volume_downsampled, seqs_labels_downsampled

    with open(f'downsampled_seqs_spk_vol_label_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
        pickle.dump(seed_scores, f)
        
    print(f"Volume parameter {vol_param} done.")

Processed batch: neurons 0-5000/100000
Processed batch: neurons 5000-10000/100000
Processed batch: neurons 10000-15000/100000
Processed batch: neurons 15000-20000/100000
Processed batch: neurons 20000-25000/100000
Processed batch: neurons 25000-30000/100000
Processed batch: neurons 30000-35000/100000
Processed batch: neurons 35000-40000/100000
Processed batch: neurons 40000-45000/100000
Processed batch: neurons 45000-50000/100000
Processed batch: neurons 50000-55000/100000
Processed batch: neurons 55000-60000/100000
Processed batch: neurons 60000-65000/100000
Processed batch: neurons 65000-70000/100000
Processed batch: neurons 70000-75000/100000
Processed batch: neurons 75000-80000/100000
Processed batch: neurons 80000-85000/100000
Processed batch: neurons 85000-90000/100000
Processed batch: neurons 90000-95000/100000
Processed batch: neurons 95000-100000/100000
Generated sequences for motif 0/20
Generated sequences for motif 1/20
Generated sequences for motif 2/20
Generated sequences 

In [ ]:
## SAVING ONLY ZSCORES ## 
seed_scores = {}
params["n_neurons"] = 100000
seed_down_seqs = {}
seed_down_spk_times = {}
seed_down_vol = {}

for seed in range(50):
    seqs, seqs_labels, spk_times, sequences, true_templates, mu, sigma, volume, _, _ = simulate_sequences(**params, random_state=seed, plot=False, batch_size=5000)
    seqs_downsampled, spk_times_downsampled, volume_downsampled = downsample_sequences(sequences, spk_times, volume, params["n_neurons"], n_neurons_keep=100, min_length=5, random_state=1)

    # get the seqs_labels for the downsampled sequences
    seqs_labels_downsampled = []
    for seq in seqs_downsampled:
        for i, template in enumerate(true_templates):
            if np.array_equal(seq, template):
                seqs_labels_downsampled.append(seqs_labels[i])
                break
    
    motif_idx_down = []
    motif_ids = [m for m in seqs_downsampled.keys()] # if len(seqs_downsampled[m]) > 20] 

    seqs_flat = []
    for motif_id in motif_ids:
        motif_idx_down.extend([motif_id] * len(seqs_downsampled[motif_id]))
        seqs_flat.extend(seqs_downsampled[motif_id])
        seed_down_seqs[seed] = seqs_downsampled[motif_id]
        seed_down_spk_times[seed] = spk_times_downsampled[motif_id]
        seed_down_vol[seed] = volume_downsampled[motif_id]


    motif_idx_down = np.array(motif_idx_down)
    unique_motifs_down = np.unique(motif_idx_down)

    rep_index, nsig, pval, bmat, zmat, corrmat = rs.allmot(seqs_flat, nrm)
    within_score, within_var = add_within_clust_score(motif_idx_down, zmat, bmat)
    motif_to_score = {motif: score for motif, score in zip(unique_motifs_down, within_score)}
    seed_scores[seed] = motif_to_score

    print(f"Seed {seed} done.")

    # empty the variables to save memory
    del seqs, seqs_labels, spk_times, seqs_downsampled, spk_times_downsampled, volume_downsampled, rep_index, nsig, pval, bmat, zmat, corrmat, within_score, within_var, motif_to_score

#with open(f'50seed_downsampled_zscore_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
#    pickle.dump(seed_scores, f)


In [ ]:
seed_scores = {}
params["n_neurons"] = 100
params["vol_param"] = (0.03, 0.9)
for seed in range(50):
    seqs, seqs_labels, _, _, _, _, _, _, _, _ = simulate_sequences(**params, random_state=seed, plot=False, batch_size=5000)
    rep_index, nsig, pval, bmat, zmat, corrmat = rs.allmot(seqs, nrm)
    within_score, within_var = add_within_clust_score(seqs_labels, zmat, bmat)
    unique_motifs = np.unique(seqs_labels)
    motif_to_score = {motif: score for motif, score in zip(unique_motifs, within_score)}
    seed_scores[seed] = motif_to_score
    del seqs, seqs_labels, rep_index, nsig, pval, bmat, zmat, corrmat

with open(f'50seed_orig_zscore_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump(seed_scores, f)

In [ ]:
seed_scores = {}
params["n_neurons"] = 100000
params["vol_param"] = (0.1, 0.5)
for seed in range(50):
    seqs, seqs_labels, spk_times, sequences, true_templates, mu, sigma, volume, _, _ = simulate_sequences(**params, random_state=seed, plot=False, batch_size=5000)
    seqs_downsampled, spk_times_downsampled, volume_downsampled = downsample_sequences(sequences, spk_times, volume, params["n_neurons"], n_neurons_keep=100, min_length=5, random_state=1)
    
    motif_idx_down = []
    motif_ids = [m for m in seqs_downsampled.keys()] # if len(seqs_downsampled[m]) > 20] 

    seqs_flat = []
    for motif_id in motif_ids:
        motif_idx_down.extend([motif_id] * len(seqs_downsampled[motif_id]))
        seqs_flat.extend(seqs_downsampled[motif_id])

    motif_idx_down = np.array(motif_idx_down)
    unique_motifs_down = np.unique(motif_idx_down)

    rep_index, nsig, pval, bmat, zmat, corrmat = rs.allmot(seqs_flat, nrm)
    within_score, within_var = add_within_clust_score(motif_idx_down, zmat, bmat)
    motif_to_score = {motif: score for motif, score in zip(unique_motifs_down, within_score)}
    seed_scores[seed] = motif_to_score

    print(f"Seed {seed} done.")

    # empty the variables to save memory
    del seqs, seqs_labels, spk_times, seqs_downsampled, spk_times_downsampled, volume_downsampled, rep_index, nsig, pval, bmat, zmat, corrmat, within_score, within_var, motif_to_score

#with open(f'50seed_downsampled_zscore_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
#    pickle.dump(seed_scores, f)

with open(f'downsampled_seqs_spk_vol_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pkl', 'wb') as f:
    pickle.dump({'seqs_downsampled': seqs_downsampled, 'spk_times_downsampled': spk_times_downsampled, 'volume_downsampled': volume_downsampled}, f)

In [ ]:
with open(f'50seed_downsampled_zscore_100000_(0.02, 0.4)_(0.07, 0.9)_0.0.pkl', 'rb') as f:
    seed_scores = pickle.load(f)

In [ ]:
all_scores = []
for seed, motif_dict in seed_scores.items():
    for motif_id, score in motif_dict.items():
        if not np.isnan(score):  
            all_scores.append(score)

all_scores = np.array(all_scores)

# sum the number of scores above 0.4
threshold = 0.4
num_above_threshold = np.sum(all_scores > threshold)
print(f"Number of scores above {threshold}: {num_above_threshold}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define common bin edges for both plots
# This ensures a bin of 0.2 units is physically the same size in both plots
bin_edges = np.arange(0, 2.2, 0.1) 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

weights_a = np.ones_like(all_scores_orig) / len(all_scores_orig)
weights_b = np.ones_like(all_scores) / len(all_scores)


# Format y-axis as percentage
from matplotlib.ticker import PercentFormatter
ax1.yaxis.set_major_formatter(PercentFormatter(1))

# 2. Plot Dataset A
ax1.hist(all_scores_orig, bins=bin_edges, weights=weights_a, color='seagreen', edgecolor='black', alpha=0.8)
ax2.hist(all_scores, bins=bin_edges, weights=weights_b, color='salmon', edgecolor='black', alpha=0.8)
ax1.axvline(x=0.4, color='red', linestyle='--', label=f'{num_above_threshold_orig} ({num_above_threshold_orig / len(all_scores_orig) * 100:.1f}%) motif score above {threshold}')
ax2.axvline(x=0.4, color='red', linestyle='--', label=f'{num_above_threshold} ({num_above_threshold / len(all_scores) * 100:.1f}%) motif score above {threshold}')
ax1.axvline(x=np.median(all_scores_orig), color='blue', linestyle='--', label=f'Median: {np.median(all_scores_orig):.3f}')
ax2.axvline(x=np.median(all_scores), color='blue', linestyle='--', label=f'Median: {np.median(all_scores):.3f}')

ax1.set_title(f'Original N=100, motifs = {len(all_scores_orig)}')
ax1.set_xlabel('Z-score')
ax1.set_ylabel('Percentage of Motifs')
ax1.legend()

# 3. Plot Dataset B
ax2.set_title(f'Downsampled N=100k to R=100, motifs = {len(all_scores)}')
ax2.set_xlabel('Z-score')
ax2.legend()

# 4. Clean up the layout
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import pandas as pd

# turn the NaN score to 0 for plotting
all_scores_orig = np.where(np.isnan(all_scores_orig), 0, all_scores_orig)
all_scores = np.where(np.isnan(all_scores), 0, all_scores)

df = pd.DataFrame({
    'Z-score': np.concatenate([all_scores_orig, all_scores]),
    'Dataset': ['Original'] * len(all_scores_orig) + ['Downsampled'] * len(all_scores)
})

plt.figure(figsize=(8, 5))
plt.violinplot([all_scores_orig, all_scores], showmedians=True, showextrema=False)
plt.xticks([1, 2], ['Original', 'Downsampled'])
plt.ylabel('Z-score')
plt.title('Distribution of Z-scores')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import seaborn as sns
import pandas as pd
# turn the NaN score to 0 for plotting
all_scores_orig = np.where(np.isnan(all_scores_orig), 0, all_scores_orig)
all_scores = np.where(np.isnan(all_scores), 0, all_scores)

df = pd.DataFrame({
    'Mean Z-scores': np.concatenate([all_scores_orig, all_scores]),
    'Dataset': ['Original'] * len(all_scores_orig) + ['Downsampled'] * len(all_scores)
})

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
sns.violinplot( 
    x = 'Dataset',
    y='Mean Z-scores', 
    hue='Dataset', 
    data=df, 
    inner='quartile',
    palette='Set2',
    alpha = 0.7,
    ax=axes[0],
    legend= False,
)
lines = axes[0].get_lines()
axes[0].set_ylabel('Mean Z-scores',fontsize=15)
axes[0].set_xlabel('Dataset',fontsize=15)

# For Dataset A (first violin), the lines are usually at indices 0, 1, 2
# For Dataset B (second violin), they are 3, 4, 5
median_a_val = lines[1].get_ydata()[0]
median_b_val = lines[4].get_ydata()[0]

print(f"Original Median: {median_a_val:.3f}")
print(f"Downsampled Median: {median_b_val:.3f}")

sns.violinplot(
    y='Mean Z-scores', 
    hue='Dataset', 
    data=df, 
    split=True, 
    palette='Set2',
    alpha = 0.7,
    inner='quartile',
    ax=axes[1],
)
axes[1].set_ylabel('Mean Z-scores',fontsize=15)

plt.suptitle(f'Distribution of Mean Z-scores\nOriginal N=100 vs Downsampled N=100k to R=100\nσ = {params["sigma_range"]} v= {params["vol_param"]}', fontsize=16)
plt.tight_layout()
plt.savefig(f'50seed_zscore_comparison_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.jpg', dpi=300)
plt.savefig(f'50seed_zscore_comparison_{params["n_neurons"]}_{params["sigma_range"]}_{params["vol_param"]}_{params["rho_vol"]}.pdf', dpi=300)
plt.show()

In [ ]:
# Try this to see the "tail" behavior more clearly
sns.boxenplot(data=df, x='Dataset', y='Z-score', hue='Dataset', palette='Set2')

In [ ]:
for seed, motif_dict in seed_scores.items():
    for motif_id, score in motif_dict.items():
        if np.isnan(score):
            print(f"Seed {seed}, Motif {motif_id} has NaN score.")


In [ ]:
for seed, motif_dict in seed_scores.items():
    for motif_id, score in motif_dict.items():
        if score >= 1.96:
            print(f"Seed {seed}, Motif {motif_id}: {score:.2f}")

In [ ]:
for seed, motif_dict in seed_scores_100.items():
    for motif_id, score in motif_dict.items():
        if score >= 1.96:
            print(f"Seed {seed}, Motif {motif_id}")